#### Bronze destination table has the complete history table 

In [0]:
# spark.sql('show tables in syntheaproject.bronze')

In [0]:
tables = ['allergies','careplans','conditions','devices','encounters','imaging_studies','immunizations','medications','observations', 'organizations','patients','payer_transitions','payers','procedures','providers','supplies']
for table in tables:
    tbl = f"syntheaproject.bronze.{table}"
    spark.sql(f"alter table {tbl} set TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"CDF enabled for {tbl}")
    

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
table_configs = {
    "patients": {
        "type": "entity",
        "pk_cols": ["pat_id"],
        "rename_map": {"id": "pat_id", "birthdate": "birth_date", "deathdate": "death_date", "drivers": "drivers_license", "passport": "passport_no", "zip": "zip_code", "first": "first_name", "last": "last_name", "maiden": "maiden_name"},
        "cast_map": {"zip_code": "string"},
        "drop_cols": ["sourcefile", "is_active", "updated_at"],
        "required_cols": ["pat_id"]
    },

    "providers": {
        "type": "entity",
        "pk_cols": ["prov_id"],
        "rename_map": {"id": "prov_id", "organization": "organization_id","speciality": "speciality", "zip": "zip_code"},
        "cast_map": {"zip_code": "string"},
        "drop_cols": ["sourcefile", "is_active", "updated_at"],
        "required_cols": ["prov_id"]
    },

    "organizations": {
        "type": "entity",
        "pk_cols": ["org_id"],
        "rename_map": {"id": "org_id", "zip": "zip_code"},
        "cast_map": {"zip_code": "string"},
        "drop_cols": ["sourcefile", "is_active", "updated_at"],
        "required_cols": ["org_id"]
    },

    "payers": {
        "type": "entity",
        "pk_cols": ["pay_id"],
        "rename_map": {"id": "pay_id", "state_headquartered": "state_headquartered", "zip": "zip_code"},
        "cast_map": {"zip_code": "string"},
        "drop_cols": ["sourcefile", "is_active", "updated_at"],
        "required_cols": ["pay_id"]
    },

    "encounters": {
        "type": "event",
        "pk_cols": ["enc_id"],
        "rename_map": {"id": "enc_id", "patient": "pat_id", "organization": "org_id", "provider": "prov_id", "payer": "pay_id", "encounterclass": "encounter_class", "base_encounter_cost": "base_encounter_cost", "total_claim_cost": "total_claim_cost", "payer_coverage": "payer_coverage", "reasoncode": "reason_code", "reasondescription": "reason_description"},
        "cast_map": {"code": "string","reason_code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["enc_id", "pat_id"]
    },

    "careplans": {
        "type": "event",
        "pk_cols": ["careplan_id"],
        "rename_map": {"id": "careplan_id", "patient": "pat_id", "encounter": "enc_id", "reasoncode": "reason_code", "reasondescription": "reason_description"},
        "cast_map": {"code": "string", "reason_code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["careplan_id", "pat_id"]
    },

    "imaging_studies": {
        "type": "event",
        "pk_cols": ["imaging_study_id"],
        "rename_map": {"id": "imaging_study_id", "patient": "pat_id", "encounter": "enc_id", "bodysite_code": "bodysite_code", "bodysite_description": "bodysite_description", "modality_code": "modality_code", "modality_description": "modality_description", "sop_code": "sop_code", "sop_description": "sop_description"},
        "cast_map": {"bodysite_code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["imaging_study_id", "pat_id"]
    },

    "allergies": {
        "type": "event",
        "pk_cols": ["allergy_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "allergy_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id", "reasoncode": "reason_code", "reasondescription": "reason_description"},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "observations": {
        "type": "event",
        "pk_cols": ["observation_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "observation_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id", "value": "value", "units": "units", "type": "observation_type"},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "procedures": {
        "type": "event",
        "pk_cols": ["procedure_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "procedure_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id", "reasoncode": "reason_code", "reasondescription": "reason_description"},
        "cast_map": {"code": "string", "reason_code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "conditions": {
        "type": "event",
        "pk_cols": ["condition_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "condition_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id"},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "immunizations": {
        "type": "event",
        "pk_cols": ["immunization_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "immunization_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id"},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "medications": {
        "type": "event",
        "pk_cols": ["medication_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "medication_sk",
        "rename_map": {"patient": "pat_id", "payer": "pay_id", "encounter": "enc_id", "payer_coverage": "payer_coverage", "totalcost": "total_cost", "reasoncode": "reason_code", "reasondescription": "reason_description"},
        "cast_map": {"code": "string","reason_code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "supplies": {
        "type": "event",
        "pk_cols": ["supply_sk"],
        "business_key_cols": ["pat_id", "enc_id", "code"],
        "hash_key_col": "supply_sk",
        "rename_map": {"patient": "pat_id", "encounter": "enc_id"},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "enc_id", "code"]
    },

    "devices": {
        "type": "event",
        "pk_cols": ["dev_id"],
        "rename_map": {"patient": "pat_id","encounter": "enc_id", "udi" : 'dev_id'},
        "cast_map": {"code": "string"},
        "drop_cols": ["sourcefile"],
        "required_cols": ["patient_id", "encounter_id"]
    },

    "payer_transitions": {
        "type": "event",
        "pk_cols": ["pay_trans_sk"],
        "business_key_cols": ["pat_id", "pay_id", "start_year"],
        "hash_key_col": "pay_trans_sk",
        "rename_map": {"patient": "pat_id","payer": "pay_id"},
        "cast_map": {},
        "drop_cols": ["sourcefile"],
        "required_cols": ["pat_id", "pay_id", "start_year"]
    }
}


In [0]:
def normalize_columns_to_lower(df):
# Lowercase every column name.
    for c in df.columns:
        df = df.withColumnRenamed(c,c.lower())
    return df
def table_exists(full_table_name):
    return spark.catalog.tableExists(full_table_name)

In [0]:
def rename_columns(df, rename_map):
    for old_col, new_col in rename_map.items():
        if old_col in df.columns:
            df = df.withColumnRenamed(old_col, new_col)
    return df
def drop_columns_if_present(df, cols_to_drop):
    for c in cols_to_drop:
        if c in df.columns:
            df = df.drop(c)
    return df
def cast_columns(df, cast_map):
    for col_name, dtype in cast_map.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(dtype))
    return df

In [0]:
def add_bronze_ingested_at_alias(df):
    if "ingested_at" in df.columns:
        df = df.withColumnRenamed("ingested_at", "bronze_ingested_at")
    return df
def add_silver_processed_at(df):
    return df.withColumn("silver_processed_at", F.current_timestamp())

In [0]:
def clean_string_columns(df):
    for c, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
    return df
def validate_required_columns(df, required_cols):
    for c in required_cols:
        if c in df.columns:
            df = df.filter(F.col(c).isNotNull())
    return df

In [0]:
def add_hash_key(df, hash_col, key_cols):
    return df.withColumn(
        hash_col, 
        F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("__NULL__")) for c in key_cols]),256)
    )
def dedupe_by_keys(df, key_cols, order_cols=None):
    if not isinstance(key_cols, list):
        key_cols = [key_cols]
    if order_cols and len(order_cols) > 0:
        window_spec = Window.partitionBy(*key_cols).orderBy(*[F.col(c).desc_nulls_last() for c in order_cols])
        return (df.withColumn("rn", F.row_number().over(window_spec)).filter(F.col("rn") == 1).drop("rn"))
    else:
        return df.dropDuplicates(key_cols)

In [0]:
def clean_string_columns(df):
    for c, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
    return df
def validate_required_columns(df, required_cols):
    for c in required_cols:
        if c in df.columns:
            df = df.filter(F.col(c).isNotNull())
    return df

In [0]:
def build_merge_condition(pk_cols):
    if isinstance(pk_cols, list):
        return " AND ".join([f"target.{c} = source.{c}" for c in pk_cols])
    return f"target.{pk_cols} = source.{pk_cols}"
def latest(full_table_name):
    hist = spark.sql(f"describe history {full_table_name}")
    latest_version = hist.select(F.max("version").alias("v")).collect()[0]["v"]
    return int(latest_version)

In [0]:
def curated_df(full_bronze_table, cfg):
    # Used for initial silver creation from current bronze snapshot.
    df = spark.read.table(full_bronze_table)
    df = normalize_columns_to_lower(df)
    # print(df.columns)
    df = rename_columns(df, cfg.get("rename_map", {}))
    # print(df.columns)
    df = add_bronze_ingested_at_alias(df)
    df = drop_columns_if_present(df, cfg.get("drop_cols", []))
    df = clean_string_columns(df)

    if "business_key_cols" in cfg and "hash_key_col" in cfg:
        df = add_hash_key(df, cfg["hash_key_col"], cfg["business_key_cols"])
    df = validate_required_columns(df, cfg.get("required_cols", []))
    df = cast_columns(df, cfg.get("cast_map", {}))
    df = add_silver_processed_at(df)
    return df

In [0]:

def LoadInit(table_name, cfg):
    bronze_table = f"syntheaproject.bronze.{table_name}"
    silver_table = f"syntheaproject.silver.{table_name}"
    if not table_exists(silver_table):
        df = curated_df(bronze_table, cfg)
        if cfg["type"] == "entity":
            if "is_active" in spark.read.table(bronze_table).columns:
                df = df.filter(F.col("is_active") == True)
            df = dedupe_by_keys(df, cfg["pk_cols"], order_cols=["bronze_ingested_at"])
        else:
            if "business_key_cols" in cfg:
                df = dedupe_by_keys(df, cfg["pk_cols"], order_cols=["bronze_ingested_at"])
            else:
                df = dedupe_by_keys(df, cfg["pk_cols"], order_cols=["bronze_ingested_at"])
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table))

In [0]:
def process_entity(table_name, cfg):
    bronze_table = f"syntheaproject.bronze.{table_name}"
    silver_table = f"syntheaproject.silver.{table_name}"
    checkpoint_path = f"/Volumes/syntheaproject/default/syntheavolume/_checkpoints/silver/{table_name}"
    pk_cols = cfg["pk_cols"]
    latest_version = latest(bronze_table)
    source_stream = (spark.readStream.option("readChangeFeed", "true").option("startingVersion", latest_version).table(bronze_table))
    source_stream = source_stream.filter(F.col("_change_type").isin("insert", "update_postimage"))

    def upsert(batch_df, batch_id):
        if batch_df.isEmpty():
            return
        df = batch_df.drop("_change_type", "_commit_version", "_commit_timestamp")
        df = normalize_columns_to_lower(df)
        df = rename_columns(df, cfg.get("rename_map", {}))
        df = add_bronze_ingested_at_alias(df)
        df = drop_columns_if_present(df, cfg.get("drop_cols", []))
        df = clean_string_columns(df)

        if "is_active" in df.columns:
            df = df.filter(F.col("is_active") == True)
        df = validate_required_columns(df, cfg.get("required_cols", []))
        df = cast_columns(df, cfg.get("cast_map", {}))
        df = add_silver_processed_at(df)
        merge_condition = build_merge_condition(pk_cols)
        df = dedupe_by_keys(df, pk_cols, order_cols=["bronze_ingested_at"])
        df.createOrReplaceTempView("updates")

        if table_exists(silver_table):
            spark.sql(f"MERGE INTO {silver_table} AS target USING updates AS source ON {merge_condition} WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
        else:
            (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table))

    query = (source_stream.writeStream.foreachBatch(upsert).option("checkpointLocation", checkpoint_path).trigger(availableNow=True).start())
    query.awaitTermination()

In [0]:
def process_event(table_name, cfg):
    bronze_table = f"syntheaproject.bronze.{table_name}"
    silver_table = f"syntheaproject.silver.{table_name}"
    checkpoint_path = f"/Volumes/syntheaproject/default/syntheavolume/_checkpoints/silver/{table_name}"
    latest_version = latest(bronze_table)
    source_stream = (spark.readStream.option("readChangeFeed", "true").option("startingVersion", latest_version).table(bronze_table))
    source_stream = source_stream.filter(F.col("_change_type") == "insert")

    def append_batch(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        df = batch_df.drop("_change_type", "_commit_version", "_commit_timestamp")
        df = normalize_columns_to_lower(df)
        df = rename_columns(df, cfg.get("rename_map", {}))
        df = add_bronze_ingested_at_alias(df)
        df = drop_columns_if_present(df, cfg.get("drop_cols", []))
        df = clean_string_columns(df)

        # creatikng hash surrogate key for composite-key tables
        if "business_key_cols" in cfg and "hash_key_col" in cfg:
            df = add_hash_key(df, cfg["hash_key_col"], cfg["business_key_cols"])
            df = dedupe_by_keys(df, cfg["hash_key_col"], order_cols=["bronze_ingested_at"])
        else:
            df = dedupe_by_keys(df, cfg["pk_cols"], order_cols=["bronze_ingested_at"])

        df = validate_required_columns(df, cfg.get("required_cols", []))
        df = cast_columns(df, cfg.get("cast_map", {}))
        df = add_silver_processed_at(df)

        if not table_exists(silver_table):
            (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table))
        else:
            (df.write.format("delta").mode("append").saveAsTable(silver_table))
    query = (source_stream.writeStream.foreachBatch(append_batch).option("checkpointLocation", checkpoint_path).trigger(availableNow=True).start())
    query.awaitTermination()

In [0]:
for table_name, cfg in table_configs.items():
    print(f"Processing table: {table_name}")
    print(f"Type: {cfg['type']}")
    LoadInit(table_name, cfg)

    if cfg["type"] == "entity":
        process_entity(table_name, cfg)
    else:
        process_event(table_name, cfg)

In [0]:
# Transformations: (datatype fixes, trimming, null handling, deduplication, standardization, remove unnecessary columns, derive helper fields)